# One-click GPU voice trainer — upload once, download one ZIP

**Run the single cell below.** Upload a recording 2 seconds–2 hours long. The script examines at most its first 3 minutes, selects up to 30 seconds of usable reference audio, automatically creates **400 accepted original/converted teacher pairs**, trains a **full LLVC HiFi-GAN model** for 5,000 updates by default, validates ONNX, and downloads **one ZIP containing the `.pth` and `.onnx` files**, configuration, and A/B samples.

**Requirements:** Colab **GPU (T4)**. Initial setup and training can take hours. You can change `STEPS = 5000` at the top of the cell. Uses one isolated Python 3.10 environment; no local Chromebook Python. Audio longer than 3 minutes is not processed past minute 3, but the full file must be uploaded first. 2–3 second samples are allowed but may produce poor voice quality. GPU time and Colab sessions aren't guaranteed.

The generated ONNX executes in ONNX Runtime, but voice similarity and real-time performance still require listening/testing. The pipeline has not yet been tested end-to-end as one integrated run. Upload only audio you have permission to process.


In [ ]:
# ONE RUN: upload audio -> 400 valid voice pairs -> train PTH -> export ONNX -> download ONE ZIP.
# Supports 2 seconds–2 hours of input audio; inspects at most the first 3 minutes.
# Change STEPS below before running if desired; 5000 uses the prior working training length.
PAIRS = 400
STEPS = 5000
BATCH = 2
TEACHER_STEPS = 10

from pathlib import Path
from google.colab import files
import json
import os
import shutil
import subprocess
import sys
import uuid
import zipfile

print("Select a WAV, MP3, FLAC, M4A, OGG, or OPUS voice recording.", flush=True)
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one voice recording.")
name = next(iter(uploaded))
source = (Path.cwd() / name).resolve()
assert source.is_file(), f"Uploaded audio missing: {source}"
job_id = uuid.uuid4().hex[:10]
uploads = Path("/content/voice-uploads") / job_id
uploads.mkdir(parents=True, exist_ok=True)
audio = uploads / source.name
shutil.move(str(source), str(audio))
# Free Colab's upload dict memory before loading the GPU models.
del uploaded

def run(*cmd):
    command = [str(item) for item in cmd]
    print("+", " ".join(command), flush=True)
    subprocess.run(command, check=True)

repo = Path("/content/irl-voice")
if not (repo / ".git").is_dir():
    run("git", "clone", "--depth", "1",
        "https://github.com/tbenitz/IRL-Voice-Changer.git", repo)
else:
    run("git", "-C", repo, "pull", "--ff-only")

python = Path("/content/voice-venv/bin/python")
ready = False
if python.is_file() and Path("/content/seed-vc/inference.py").is_file() and Path("/content/llvc-upstream/train.py").is_file():
    check = subprocess.run([
        str(python), "-c",
        "import torch,onnx,onnxruntime,soundfile,librosa;"
        "assert torch.cuda.is_available()"
    ], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    ready = (check.returncode == 0)
if not ready:
    print("Installing or repairing the GPU training environment...", flush=True)
    run(sys.executable, repo / "voice-training" / "colab_setup.py")

run_dir = Path("/content/voice-jobs") / job_id
run(str(python), repo / "voice-training" / "one_click_pipeline.py",
    "--audio", audio, "--run-dir", run_dir,
    "--repo", repo,
    "--seed-vc", "/content/seed-vc",
    "--llvc", "/content/llvc-upstream",
    "--cache", "/content/voice-cache",
    "--pairs", PAIRS,
    "--steps", STEPS,
    "--batch", BATCH,
    "--teacher-steps", TEACHER_STEPS)

finished = list(run_dir.glob("*_PTH_ONNX.zip"))
if len(finished) != 1:
    raise RuntimeError(f"Finished archive was not produced in {run_dir}")
bundle = finished[0]
with zipfile.ZipFile(bundle) as package:
    assert package.testzip() is None, "ZIP file integrity check failed."
    assert any(x.endswith(".onnx") for x in package.namelist())
    assert any(x.endswith(".pth") for x in package.namelist())
print("\nSUCCESS! Downloading ONE ZIP with PTH + ONNX + training report.", flush=True)
files.download(str(bundle))
